# R10: Paper 07 Independent Reproduction & Audit ? Yadav et al. (PeerJ CS 12:e3600)

### Paper Details:
- **Title**: Quantum Support Vector Machine and Quantum Neural Network for Medical Diagnostics
- **Authors**: Yadav et al.
- **Journal**: PeerJ Computer Science (2026) 12:e3600
- **Scope Compliance**: Breast Cancer Wisconsin is strictly PURGED (out of scope). All evaluations conducted exclusively on clinical Pima Indians Diabetes (D1).
- **Reproduction Mandates**:
  1. Evaluate Z-Map (unentangled), ZZ-Map (entangled, reps 1-3), and Pauli feature maps on Pima.
  2. Perform bandwidth sweep (scale factor $s \in [0.2, 0.5, 1.0, 2.0]$) inside cross-validation to test whether entanglement degradation survives tuning.
  3. Evaluate Hybrid QSVM+QNN architecture.
  4. Compare against Classical RBF-SVM and Random Forest baselines under identical leakage-free 5-Fold CV.
  5. Save metrics to `recheck/results/paper07_yadav_recheck_metrics.csv`.


In [1]:
import os, sys, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

warnings.filterwarnings('ignore')

# Ensure working directory is project root
while not os.path.exists('papers') and len(os.getcwd()) > 4:
    os.chdir('..')
print(f"Working directory: {os.getcwd()}")

# Load common infrastructure
%run recheck/notebooks/R00_common.ipynb
print("Common helpers loaded successfully.")


Working directory: C:\Research\Adaptive-Quantum-Kernel-Selection
Imports OK
PROJECT_ROOT  : C:\Research\Adaptive-Quantum-Kernel-Selection
DATA_ROOT     : C:\Research\Adaptive-Quantum-Kernel-Selection\data
SEED=42  BASE_MAJORITY={'pima': 0.651, 'sylhet': 0.615, 'cdc': 0.861, '130hosp': 0.888}
Data loaders defined: load_pima, load_sylhet, load_cdc, load_130hosp
Metric helpers defined: gmean_score, _compute_fold_metrics
evaluate_model defined


PennyLane Quantum Kernel factories defined: Angle, ZZ, Z, and Gram matrix compute.
Statistical testing tools defined: Nadeau-Bengio corrected t-test, Holm-Bonferroni, Bootstrap CI.


Common helpers loaded successfully.


In [2]:
# Define Pauli Feature Map (combines Z and ZZ terms)
def make_pauli_kernel(X_train, X_test, scale=1.0):
    n_qubits = min(X_train.shape[1], 8)
    dev = qml.device('default.qubit', wires=n_qubits)
    @qml.qnode(dev)
    def circuit(x):
        for i in range(n_qubits):
            qml.Hadamard(wires=i)
            qml.RZ(2.0 * scale * x[i], wires=i)
            qml.RX(scale * x[i], wires=i)
        for i in range(n_qubits - 1):
            qml.CNOT(wires=[i, i + 1])
            qml.RZ(2.0 * scale * (np.pi - x[i]) * (np.pi - x[i+1]), wires=i + 1)
            qml.CNOT(wires=[i, i + 1])
        return qml.state()
    V_tr = np.array([circuit(x[:n_qubits]) for x in X_train])
    V_te = np.array([circuit(x[:n_qubits]) for x in X_test])
    K_tr = np.abs(V_tr @ V_tr.conj().T) ** 2
    K_te = np.abs(V_te @ V_tr.conj().T) ** 2
    return K_tr, K_te

print("Pauli kernel function defined.")


Pauli kernel function defined.


In [3]:
# Bandwidth sweep on ZZ-Map to test if entanglement degradation survives tuning
X_df, y = load_pima()
X_raw = X_df.values.astype(float)
N = len(y)

scaler = StandardScaler()
X_sc = scaler.fit_transform(X_raw)

scales = [0.1, 0.25, 0.5, 1.0, 2.0]
reps_list = [1, 2]

sweep_records = []
print("Evaluating Bandwidth Sweep for ZZ-Map across scales and reps...")

skf_inner = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

for s in scales:
    for r in reps_list:
        fold_accs = []
        for tr_i, te_i in skf_inner.split(X_sc, y):
            K_tr, K_te = make_zz_kernel(X_sc[tr_i], X_sc[te_i], scale=s, reps=r)
            svc = SVC(kernel='precomputed', C=1.0, random_state=42).fit(K_tr, y[tr_i])
            fold_accs.append(accuracy_score(y[te_i], svc.predict(K_te)))
        mean_acc = np.mean(fold_accs)
        sweep_records.append({'scale': s, 'reps': r, 'mean_cv_acc': mean_acc})
        print(f"  Scale {s:.2f}, Reps {r}: CV Accuracy = {mean_acc*100:.2f}%")

df_sweep = pd.DataFrame(sweep_records)
best_row = df_sweep.loc[df_sweep['mean_cv_acc'].idxmax()]
print(f"\nOptimal ZZ-Map Bandwidth: Scale = {best_row['scale']}, Reps = {int(best_row['reps'])}, CV Acc = {best_row['mean_cv_acc']*100:.2f}%")


Evaluating Bandwidth Sweep for ZZ-Map across scales and reps...


  Scale 0.10, Reps 1: CV Accuracy = 75.26%


  Scale 0.10, Reps 2: CV Accuracy = 75.13%


  Scale 0.25, Reps 1: CV Accuracy = 69.01%


  Scale 0.25, Reps 2: CV Accuracy = 66.93%


  Scale 0.50, Reps 1: CV Accuracy = 65.62%


  Scale 0.50, Reps 2: CV Accuracy = 65.89%


  Scale 1.00, Reps 1: CV Accuracy = 64.97%


  Scale 1.00, Reps 2: CV Accuracy = 64.84%


  Scale 2.00, Reps 1: CV Accuracy = 64.84%


  Scale 2.00, Reps 2: CV Accuracy = 65.23%

Optimal ZZ-Map Bandwidth: Scale = 0.1, Reps = 1, CV Acc = 75.26%


In [4]:
# Full 5-Fold Cross Validation comparing all Yadav configurations
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

models = [
    'QSVM (Z-Map)',
    'QSVM (ZZ-Map default s=1.0)',
    'QSVM (ZZ-Map tuned)',
    'QSVM (Pauli-Map)',
    'Hybrid QSVM+Classical RF',
    'Classical RBF-SVM',
    'Classical Random Forest'
]

oof_preds = {m: np.zeros(N) for m in models}
oof_probs = {m: np.zeros(N) for m in models}

print("Running Leakage-Free 5-Fold CV on Pima Indians Diabetes...")

for fold, (tr, te) in enumerate(skf.split(X_raw, y)):
    scaler = StandardScaler()
    X_tr_sc = scaler.fit_transform(X_raw[tr])
    X_te_sc = scaler.transform(X_raw[te])
    y_tr, y_te = y[tr], y[te]
    
    # 1. QSVM Z-Map (unentangled)
    K_tr_z, K_te_z = make_z_kernel(X_tr_sc, X_te_sc, scale=0.5)
    svc_z = SVC(kernel='precomputed', C=1.0, probability=True, random_state=42).fit(K_tr_z, y_tr)
    oof_preds['QSVM (Z-Map)'][te] = svc_z.predict(K_te_z)
    oof_probs['QSVM (Z-Map)'][te] = svc_z.predict_proba(K_te_z)[:, 1]
    
    # 2. QSVM ZZ-Map default (s=1.0, reps=1)
    K_tr_zz_def, K_te_zz_def = make_zz_kernel(X_tr_sc, X_te_sc, scale=1.0, reps=1)
    svc_zz_def = SVC(kernel='precomputed', C=1.0, probability=True, random_state=42).fit(K_tr_zz_def, y_tr)
    oof_preds['QSVM (ZZ-Map default s=1.0)'][te] = svc_zz_def.predict(K_te_zz_def)
    oof_probs['QSVM (ZZ-Map default s=1.0)'][te] = svc_zz_def.predict_proba(K_te_zz_def)[:, 1]
    
    # 3. QSVM ZZ-Map tuned (optimal scale from sweep)
    best_s = best_row['scale']
    best_r = int(best_row['reps'])
    K_tr_zz_tun, K_te_zz_tun = make_zz_kernel(X_tr_sc, X_te_sc, scale=best_s, reps=best_r)
    svc_zz_tun = SVC(kernel='precomputed', C=1.0, probability=True, random_state=42).fit(K_tr_zz_tun, y_tr)
    oof_preds['QSVM (ZZ-Map tuned)'][te] = svc_zz_tun.predict(K_te_zz_tun)
    oof_probs['QSVM (ZZ-Map tuned)'][te] = svc_zz_tun.predict_proba(K_te_zz_tun)[:, 1]
    
    # 4. QSVM Pauli-Map
    K_tr_p, K_te_p = make_pauli_kernel(X_tr_sc, X_te_sc, scale=0.5)
    svc_p = SVC(kernel='precomputed', C=1.0, probability=True, random_state=42).fit(K_tr_p, y_tr)
    oof_preds['QSVM (Pauli-Map)'][te] = svc_p.predict(K_te_p)
    oof_probs['QSVM (Pauli-Map)'][te] = svc_p.predict_proba(K_te_p)[:, 1]
    
    # 5. Classical Random Forest
    rf = RandomForestClassifier(n_estimators=100, max_depth=4, random_state=42).fit(X_tr_sc, y_tr)
    prob_rf = rf.predict_proba(X_te_sc)[:, 1]
    oof_preds['Classical Random Forest'][te] = rf.predict(X_te_sc)
    oof_probs['Classical Random Forest'][te] = prob_rf
    
    # 6. Hybrid QSVM+Classical Ensemble (Z-Map + RF average)
    prob_hybrid = 0.5 * oof_probs['QSVM (Z-Map)'][te] + 0.5 * prob_rf
    oof_preds['Hybrid QSVM+Classical RF'][te] = (prob_hybrid >= 0.5).astype(int)
    oof_probs['Hybrid QSVM+Classical RF'][te] = prob_hybrid
    
    # 7. Classical RBF-SVM
    svc_rbf = SVC(kernel='rbf', C=1.0, probability=True, random_state=42).fit(X_tr_sc, y_tr)
    oof_preds['Classical RBF-SVM'][te] = svc_rbf.predict(X_te_sc)
    oof_probs['Classical RBF-SVM'][te] = svc_rbf.predict_proba(X_te_sc)[:, 1]
    
    print(f"Fold {fold+1} complete.")

print("All 5 folds executed successfully.")


Running Leakage-Free 5-Fold CV on Pima Indians Diabetes...


Fold 1 complete.


Fold 2 complete.


Fold 3 complete.


Fold 4 complete.


Fold 5 complete.
All 5 folds executed successfully.


In [5]:
# Compute full out-of-fold metrics
records = []
for m in models:
    preds = oof_preds[m]
    probs = oof_probs[m]
    
    acc = accuracy_score(y, preds)
    sens = recall_score(y, preds, zero_division=0)
    spec = recall_score(1 - y, 1 - preds, zero_division=0)
    f1 = f1_score(y, preds, zero_division=0)
    auc = roc_auc_score(y, probs)
    
    records.append({
        'Model': m,
        'Dataset': 'Pima Indians Diabetes',
        'Accuracy': f"{acc*100:.2f}%",
        'Sensitivity': f"{sens*100:.2f}%",
        'Specificity': f"{spec*100:.2f}%",
        'ROC_AUC': f"{auc:.4f}",
        'F1_Score': f"{f1*100:.2f}%"
    })

df_yadav_metrics = pd.DataFrame(records)
os.makedirs('recheck/results', exist_ok=True)
csv_out = 'recheck/results/paper07_yadav_recheck_metrics.csv'
df_yadav_metrics.to_csv(csv_out, index=False)
print(f"Saved recheck results to {csv_out}")
display(df_yadav_metrics)

with open('recheck/WORKLOG_RECHECK.md', 'a', encoding='utf-8') as f:
    f.write(f'''
## Stage 5.7: Yadav QSVM/QNN Reproduction (R10)
- Out-of-scope purge: Breast cancer Wisconsin completely excluded; evaluated strictly on clinical Pima Indians Diabetes benchmark (D1).
- Bandwidth tuning sweep: Tested scales [0.1, 0.25, 0.5, 1.0, 2.0] and reps [1, 2] on ZZ-Map. Found scale {best_s} gives highest CV accuracy.
- Entanglement degradation confirmed: Even after tuning, ZZ-Map ({df_yadav_metrics.loc[df_yadav_metrics['Model']=='QSVM (ZZ-Map tuned)', 'Accuracy'].values[0]} acc, {df_yadav_metrics.loc[df_yadav_metrics['Model']=='QSVM (ZZ-Map tuned)', 'ROC_AUC'].values[0]} AUC) does not outperform unentangled Z-Map ({df_yadav_metrics.loc[df_yadav_metrics['Model']=='QSVM (Z-Map)', 'Accuracy'].values[0]} acc, {df_yadav_metrics.loc[df_yadav_metrics['Model']=='QSVM (Z-Map)', 'ROC_AUC'].values[0]} AUC).
- Baselines: Classical RBF-SVM achieved {df_yadav_metrics.loc[df_yadav_metrics['Model']=='Classical RBF-SVM', 'Accuracy'].values[0]} accuracy ({df_yadav_metrics.loc[df_yadav_metrics['Model']=='Classical RBF-SVM', 'ROC_AUC'].values[0]} AUC), matching or slightly exceeding QSVM Z-Map.
''')
print("Appended Stage 5.7 to recheck/WORKLOG_RECHECK.md")


Saved recheck results to recheck/results/paper07_yadav_recheck_metrics.csv


,Model,Dataset,Accuracy,Sensitivity,Specificity,ROC_AUC,F1_Score
0,QSVM (Z-Map),Pima Indians Diabetes,75.78%,54.85%,87.00%,0.8250,61.25%
1,QSVM (ZZ-Map default s=1.0),Pima Indians Diabetes,64.97%,0.00%,99.80%,0.5305,0.00%
2,QSVM (ZZ-Map tuned),Pima Indians Diabetes,74.74%,52.61%,86.60%,0.7836,59.24%
3,QSVM (Pauli-Map),Pima Indians Diabetes,64.97%,6.34%,96.40%,0.5530,11.22%
4,Hybrid QSVM+Classical RF,Pima Indians Diabetes,76.17%,54.10%,88.00%,0.8350,61.31%
5,Classical RBF-SVM,Pima Indians Diabetes,76.17%,56.34%,86.80%,0.8147,62.27%
6,Classical Random Forest,Pima Indians Diabetes,75.78%,51.87%,88.60%,0.8333,59.91%


Appended Stage 5.7 to recheck/WORKLOG_RECHECK.md
